#LOAD_Preparation_Data

*Propósito:* Construir la tabla maestra de features (portafolio + variables) aplicando lógica de negocio: Dueños, materialidad, variables AG, antiguedad SUNAT,etc. Escritura particionada por codmes.

In [0]:
%run ../config/config

In [0]:
import sys
import time
sys.path.insert(0, str(PROJECT_PATH))
from pyspark.sql import SparkSession
from utils.logger import MLOpsLogger
from utils.month_delta import month_delta
from utils.data_preparation.universo_implementacion import UniversoImplementacion

import mlflow
import mlflow.tracking._model_registry.utils
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks"

In [0]:
logger = MLOpsLogger(
    name='03_load_preparation_data',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO",
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '03_load_preparation_data'
    }
)

In [0]:
mlflow.set_registry_uri("databricks-uc")
logger.info(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

In [0]:
spark = SparkSession.builder.getOrCreate()

# Determinar meses a procesar (histórico o solo actual)
if FIRST_RUN and MESES_HISTORICOS_CALIDAD > 0:
    meses = []
    tmp = MES_VTA
    for _ in range(MESES_HISTORICOS_CALIDAD):
        tmp = int(month_delta(str(tmp), -1))
        meses.append(tmp)
    meses.reverse()
    meses_a_procesar = meses + [MES_VTA]
else:
    meses_a_procesar = [MES_VTA]

logger.info(f"Meses a procesar: {meses_a_procesar}")

# Nombre de la tabla de salida (incluye sufijo _v2)
OUTPUT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.{BASE_NAME_TABLE_MASTER}{TABLE_VERSION_SUFFIX}"
logger.info(f"Tabla de salida: {OUTPUT_TABLE}")

# ==========================================================
# Ejecutar pipeline
# ==========================================================
pipeline = UniversoImplementacion(spark, logger, {
    "SOURCE_TABLE": SOURCE_TABLE,
    "CATALOG_NAME": CATALOG_NAME,
    "SCHEMA_NAME": SCHEMA_NAME,
    "BASE_NAME_TABLE_MASTER": BASE_NAME_TABLE_MASTER,
    "ENV": ENV,
    "TABLE_VERSION_SUFFIX": TABLE_VERSION_SUFFIX,
    "FEATURE_NAMES": FEATURE_NAMES,
    "FIRST_RUN": FIRST_RUN,
    "MESES_HISTORICOS_CALIDAD": MESES_HISTORICOS_CALIDAD,
    "DEBUG_MLOPS": DEBUG_MLOPS,
})

try:
    start = time.time()
    resultados = pipeline.execute(meses_a_procesar, OUTPUT_TABLE)
    duration = time.time() - start
    logger.info(f"✅ Pipeline completado en {duration:.2f} segundos")

    # ========== RESUMEN FINAL ==========
    logger.info("\n" + "=" * 80)
    logger.info("RESUMEN FINAL")
    logger.info("=" * 80)
    exitosos = sum(1 for r in resultados if r['exitoso'])
    logger.info(f"Meses: {len(resultados)} | OK: {exitosos} | Error: {len(resultados) - exitosos}")
    for r in resultados:
        if r['exitoso']:
            logger.info(f"  ✅ {r['mes']}: {r['registros']:,} registros en {r['duracion']:.2f}s")
        else:
            logger.error(f"  ❌ {r['mes']}: {r.get('error', '?')}")
    # ========== FIN DEL RESUMEN ==========

except Exception as e:
    logger.log_exception(operation='load_preparation_data', exception=e, should_raise=True, mes_vta=MES_VTA, environment=ENV)
    raise

In [0]:
# ==========================================================
# VALIDACIÓN VISUAL DE LA TABLA GENERADA
# ==========================================================
logger.info("\n" + "=" * 80)
logger.info("VALIDACIÓN DE LA TABLA GENERADA")
logger.info("=" * 80)

df_validate = spark.table(OUTPUT_TABLE)

# 1. Número de columnas
num_cols = len(df_validate.columns)
logger.info(f"📊 Número total de columnas en la tabla: {num_cols}")

# 2. Lista de columnas esperadas (FEATURE_NAMES)
columnas_tabla = set(df_validate.columns)
columnas_esperadas = set(FEATURE_NAMES)
faltantes = columnas_esperadas - columnas_tabla
if faltantes:
    logger.warning(f"⚠️ Faltan {len(faltantes)} variables esperadas: {list(faltantes)}")
else:
    logger.info(f"✅ Todas las {len(FEATURE_NAMES)} variables de FEATURE_NAMES están presentes")

# 3. Mostrar una muestra de los datos (5 filas)
logger.info("👀 Muestra de los primeros 5 registros:")
display(df_validate.limit(5))

# 4. Conteo de registros por partición (codmes)
logger.info("📦 Distribución de registros por partición (codmes):")
display(df_validate.groupBy("codmes").count().orderBy("codmes"))

# 5. Guardar información en task values para los siguientes notebooks
dbutils.jobs.taskValues.set(key="features_table", value=OUTPUT_TABLE)
dbutils.jobs.taskValues.set(key="features_table_columns", value=num_cols)
dbutils.jobs.taskValues.set(key="features_total_records", value=df_validate.count())

logger.info("✅ Validación completada")

In [0]:
if 'logger' in locals():
    logger.info(f"Finalizando notebook: {logger.name}")
    logger._flush_all_handlers()
    logger.close()